In [ ]:
import pandas as pd

df_toyama = pd.read_csv("data/tenki_toyama.csv", encoding="shift_jis")
df_toyama.head()

In [ ]:
df_tokyo = pd.read_csv("data/tenki_tokyo.csv", encoding="shift_jis")
df_tokyo.head()

In [ ]:
df_toyama

In [ ]:
# 不要な列を削除
df_toyama = df_toyama.drop([
    "平均気温(℃).1", "平均気温(℃).2", "最高気温(℃).1", 
    "最高気温(℃).2", "最低気温(℃).1", "最低気温(℃).2", 
    "降水量の合計(mm).1", "降水量の合計(mm).2", "降水量の合計(mm).3", 
    "平均雲量(10分比)", "平均雲量(10分比).1", "平均雲量(10分比).2"
    ], axis=1)

df_tokyo = df_tokyo.drop([
    "平均気温(℃).1", "平均気温(℃).2", "最高気温(℃).1", 
    "最高気温(℃).2", "最低気温(℃).1", "最低気温(℃).2", 
    "降水量の合計(mm).1", "降水量の合計(mm).2", "降水量の合計(mm).3", 
    "平均雲量(10分比)", "平均雲量(10分比).1", "平均雲量(10分比).2"
    ], axis=1)

In [ ]:
# 列名を変更
df_toyama = df_toyama.rename(columns={
    "平均気温(℃)":"toyama_平均気温(℃)", 
    "最高気温(℃)":"toyama_最高気温(℃)", 
    "最低気温(℃)":"toyama_最低気温(℃)", 
    "降水量の合計(mm)":"toyama_降水量の合計(mm)"
})
df_toyama.head()


In [ ]:
# 列名を変更
df_tokyo = df_tokyo.rename(columns={
    "平均気温(℃)":"tokyo_平均気温(℃)", 
    "最高気温(℃)":"tokyo_最高気温(℃)", 
    "最低気温(℃)":"tokyo_最低気温(℃)", 
    "降水量の合計(mm)":"tokyo_降水量の合計(mm)"
})
df_tokyo.head()

In [ ]:
# tokyoの"年月日"を削除
df_tokyo = df_tokyo.drop(columns="年月日")      # df_tokyo = df_tokyo.drop("年月日", axis=1)も可
df_tokyo.head()

In [ ]:
# データの結合
df_concat = pd.concat([df_toyama, df_tokyo], axis=1)   # 横に結合

# 列の順番を変更
df_concat = df_concat[[
    "年月日",
    "toyama_平均気温(℃)", "tokyo_平均気温(℃)", 
    "toyama_最高気温(℃)", "tokyo_最高気温(℃)", 
    "toyama_最低気温(℃)", "tokyo_最低気温(℃)", 
    "toyama_降水量の合計(mm)", "tokyo_降水量の合計(mm)"
]]
df_concat.head()

In [ ]:
# 保存
df_concat.to_csv("data/tenki_toyama_tokyo.csv", encoding="shift_jis")

In [ ]:
# CSVの読み込み
df = pd.read_csv("data/tenki_toyama_tokyo.csv", encoding="shift_jis", index_col=0)  # indexの重複を消す
df.head()

In [ ]:
# 年月日を日付型に変更
df["年月日"] = pd.to_datetime(df["年月日"])
df.head()

In [ ]:
# 2026年1,2月だけ抽出
df_2026_1_2 = df[(df["年月日"] >= "2026-01-01") & (df["年月日"] < "2026-03-01")]
df_2026_1_2.head()

In [ ]:
import matplotlib.pyplot as plt
import japanize_matplotlib
import matplotlib.dates as mdates

fig = plt.figure(figsize=(6, 3))
ax = fig.add_subplot(111, xlabel="日付", ylabel="平均気温(℃)")

# 横軸を7日ごとに表示
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))

# 表示形式
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))

ax.plot(
    df_2026_1_2["年月日"],
    df_2026_1_2["toyama_平均気温(℃)"],
    label="富山",
    color="skyblue",
    marker="o"
)

ax.plot(
    df_2026_1_2["年月日"],
    df_2026_1_2["tokyo_平均気温(℃)"],
    label="東京",
    color="orange",
    marker="o"
)

plt.title("2026年")
ax.legend()             # 凡例
ax.grid(lw=0.5)         # lw;グリッド線の濃さ

In [ ]:
# 保存
import os

os.makedirs("results", exist_ok=True)                                # 出力フォルダを作成
fig.savefig("results/avg_tmp.png", dpi=300, bbox_inches="tight")     # 結果の保存, dpi;解像度, bbox_inches;余白

In [ ]:
# 散布図
fig = plt.figure(figsize=(6, 4))
ax = fig.add_subplot(111, xlabel="合計降水量[mm]", ylabel="気温[℃]")

color_label = {"0mm": "#BBBBFF", "1-10mm": "#7777FF", ">10mm": "#0000FF"}

colors = pd.cut(
    df["tokyo_降水量の合計(mm)"],
    [0, 1, 10, 100],                # 0~1, 1~10, 10~100
    right=False,
    labels=color_label.keys()       # "0mm", "1-10mm", ">10mm"の部分がkeys
)

for color, data in df.groupby(colors):
    ax.scatter(
        data["年月日"],
        data["tokyo_降水量の合計(mm)"],
        c=color_label[color],
        label=color
    )

plt.title("1年間")
ax.set_xlabel("日付")
ax.set_ylabel("降水量[mm]")
ax.legend()

plt.show()

In [ ]:
# 2026年6月だけ抽出
df_2026_6 = df[(df["年月日"] >= "2026-06-01") & (df["年月日"] < "2026-07-01")]
df_2026_6.head()

In [ ]:
# 散布図
fig = plt.figure(figsize=(6, 4))
ax = fig.add_subplot(111, xlabel="合計降水量[mm]", ylabel="気温[℃]")

# 横軸を7日ごとに表示
ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))

color_label = {"0mm": "#BBBBFF", "1-10mm": "#7777FF", ">10mm": "#0000FF"}

colors = pd.cut(
    df["tokyo_降水量の合計(mm)"],
    [0, 1, 10, 100],                # 0~1, 1~10, 10~100
    right=False,
    labels=color_label.keys()       # "0mm", "1-10mm", ">10mm"の部分がkeys
)

for color, data in df_2026_6.groupby(colors):
    ax.scatter(
        data["年月日"],
        data["tokyo_降水量の合計(mm)"],
        c=color_label[color],
        label=color
    )

plt.title("2026年")
ax.set_xlabel("日付")
ax.set_ylabel("降水量[mm]")
ax.legend()

plt.show()

In [ ]:
# ヒストグラム
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))

sns.histplot(
    df_2026_6["tokyo_降水量の合計(mm)"],
    bins=30,
    kde=False,
    color="skyblue"
)

plt.title("2026年 東京の日別降水量の分布")
plt.xlabel("降水量(mm)")
plt.ylabel("日数")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

data = [
    df_2026_6["toyama_平均気温(℃)"],
    df_2026_6["tokyo_平均気温(℃)"]
]

ax.boxplot(
    data,
    tick_labels=["富山", "東京"]
)

plt.title("2026年 6月")
ax.set_ylabel("平均気温(℃)")
ax.grid(axis="y")

plt.show()